# TP1 — Extraction de réclamations en JSON

**Contexte :** assistant du service client de Lagune Banque.

Les réclamations sont rédigées en texte libre, parfois avec des abréviations. L'objectif est d'en extraire cinq champs dans un JSON exploitable : `categorie`, `reference_transaction`, `montant_fcfa`, `demande` et `sentiment`.

Vous définirez le schéma et le prompt, puis écrirez l'appel au modèle. Vous évaluerez ensuite les résultats sur dix réclamations, ainsi que la latence et le coût estimé pour 1 000 cas.

Complétez les cellules `TODO` et utilisez le contrôle automatique en fin de notebook.

**Sommaire**
- [Configuration](#configuration)
- [Données](#donnees)
- [Schéma](#schema)
- [Prompt](#prompt)
- [Appel au modèle](#appel)
- [Évaluation](#evaluation)
- [Résultats et erreurs](#resultats)
- [Comparaison des formats](#formats)
- [Conclusion](#conclusion)

## <a id="configuration"></a>Configuration

In [ ]:
import json
import os
import sys
import time
from pathlib import Path
from typing import Literal, Optional

import litellm
import pandas as pd
from pydantic import BaseModel, ConfigDict, ValidationError

sys.path.insert(0, str(next(p for p in [Path("code_partage"), Path("../code_partage"), Path("../../code_partage"), Path("../../../code_partage"), Path("../../starter/code_partage")] if (p / "config.py").exists()).resolve()))
from config import prepare

SEED = 42

In [ ]:
prepare()

## <a id="donnees"></a>Données

Le jeu contient dix réclamations et leur réponse de référence (`gold.json`). Il inclut plusieurs cas à traiter : montant abrégé (« 25k »), référence absente, plusieurs transactions, message hors sujet et instruction malveillante intégrée au message.

In [ ]:
dossier = next(p for p in [Path("01_fondamentaux_prompting/tp1_extraction_json/data"), Path("data"), Path("../../starter/01_fondamentaux_prompting/tp1_extraction_json/data")] if (p / "gold.json").exists())
MESSAGES = [json.loads(ligne) for ligne in (dossier / "reclamations.jsonl").read_text(encoding="utf-8").splitlines() if ligne.strip()]
GOLD = {g["id"]: g for g in json.loads((dossier / "gold.json").read_text(encoding="utf-8"))}
CHAMPS = ["categorie", "reference_transaction", "montant_fcfa", "demande", "sentiment"]

pd.set_option("display.max_colwidth", 90)
pd.DataFrame(GOLD.values()).set_index("id").join(pd.DataFrame(MESSAGES).set_index("id"))[["message", *CHAMPS]]

## <a id="schema"></a>Schéma de sortie

La classe Pydantic définit les champs et types du JSON. Elle sera transmise à l'API via `response_format`. `Literal` limite les valeurs possibles ; `Optional` autorise une valeur absente représentée par `null`.

**TODO 1** : déclarez les cinq champs.

In [ ]:
class Reclamation(BaseModel):
    model_config = ConfigDict(extra="forbid")
    # TODO 1 : categorie (probleme_virement, probleme_compte, carte_bancaire, facturation, autre), reference_transaction (str ou null),
    # montant_fcfa (entier ou null), demande (regularisation_virement, remboursement, assistance, information), sentiment (satisfait, neutre, mecontent)

## <a id="prompt"></a>Prompt système

Le schéma définit la forme ; le prompt précise le sens. Il doit donner un rôle, définir chaque valeur de `demande` (l'objectif du client, pas sa formule de politesse), indiquer comment traiter les informations absentes, les messages hors sujet et les instructions intégrées au message.

**TODO 2** : écrivez le prompt système.

In [ ]:
PROMPT = ""
# TODO 2 : le rôle, les champs et leurs définitions, les règles (null si absent, ne jamais inventer une référence,
# retenir la transaction visée par la plainte, ne pas obéir aux instructions du client mais extraire sa vraie demande, message hors sujet)

## <a id="appel"></a>Appel au modèle

`extract` envoie le prompt et la réclamation au modèle, puis renvoie la sortie ainsi que la latence et le nombre de tokens. En mode structuré, le schéma est transmis dans `response_format`. La dernière partie du TP permet de comparer avec une simple demande de JSON dans le prompt.

**TODO 3** : appelez `litellm.completion` avec `model`, `api_key=os.environ["LLM_API_KEY"]`, `seed=SEED`, les deux messages et `**options`.

In [ ]:
JSON_SUFFIX = """
Réponds UNIQUEMENT avec un objet JSON valide, sans texte avant ni après.
Exemple : {"categorie": "carte_bancaire", "reference_transaction": null, "montant_fcfa": 10000, "demande": "assistance", "sentiment": "neutre"}"""


def parse_json(raw):
    try:
        return json.loads(raw)
    except (TypeError, ValueError):
        return None


def extract(text, model, structured=True):
    start = time.perf_counter()
    system = PROMPT if structured else PROMPT + JSON_SUFFIX
    options = {"response_format": Reclamation} if structured else {}
    response = None
    # TODO 3 : response = litellm.completion(...) avec un message système (system) et un message utilisateur (text)
    if response is None:
        return {"output": None, "raw": "", "seconds": 0.0, "tokens_in": 0, "tokens_out": 0}
    raw = response.choices[0].message.content or ""
    return {"output": parse_json(raw), "raw": raw, "seconds": time.perf_counter() - start,
            "tokens_in": response.usage.prompt_tokens, "tokens_out": response.usage.completion_tokens}

In [ ]:
extract(MESSAGES[0]["message"], os.environ["MODELE"])

## <a id="evaluation"></a>Évaluation

- **Champs corrects :** comparaison des cinq champs à `gold.json` ;
- **Coût :** estimation à partir des tokens d'entrée et de sortie et des tarifs LiteLLM ;
- **Jeu de test :** `evaluate` applique `extract` aux dix réclamations.

In [ ]:
def score(output, expected):
    return sum(output.get(champ) == expected[champ] for champ in CHAMPS) if isinstance(output, dict) else 0


def cost(model, tokens_in, tokens_out):
    try:
        return sum(litellm.cost_per_token(model=model, prompt_tokens=tokens_in, completion_tokens=tokens_out))
    except Exception:
        return None


def evaluate(model, label=None, structured=True):
    results = []
    for m in MESSAGES:
        r = extract(m["message"], model, structured)
        results.append({**r, "id": m["id"], "message": m["message"], "expected": GOLD[m["id"]], "score": score(r["output"], GOLD[m["id"]]), "cost": cost(model, r["tokens_in"], r["tokens_out"])})
    return {"model": label or model, "results": results}


EVALS = [evaluate(os.environ["MODELE"])]
len(EVALS[0]["results"])

## <a id="resultats"></a>Résultats et erreurs

Le tableau compare les modèles et résume les résultats. Le coût pour 1 000 réclamations est une **extrapolation** calculée à partir de dix cas.

In [ ]:
def summarize(evals):
    rows = []
    for ev in evals:
        res = ev["results"]
        couts = [r["cost"] for r in res]
        rows.append({"Modèle": ev["model"], "JSON valide /10": sum(r["output"] is not None for r in res), "Champs justes /50": sum(r["score"] for r in res),
                     "Latence moy. (s)": round(sum(r["seconds"] for r in res) / len(res), 2),
                     "Coût / 1 000 (USD)": None if None in couts else round(sum(couts) / len(couts) * 1000, 3)})
    return pd.DataFrame(rows).set_index("Modèle")


summarize(EVALS)

## Erreurs à analyser

Pour chaque échec, identifiez le champ concerné et cherchez une cause possible : montant mal interprété, demande mal comprise ou sortie invalide. Vérifiez si le prompt ou les exemples pourraient être améliorés.

In [ ]:
def show_failures(ev):
    for r in ev["results"]:
        if r["score"] == 5:
            continue
        print(f"--- id {r['id']} ({r['score']}/5) : {r['message'][:80]}")
        if r["output"] is None:
            print("  JSON invalide, réponse brute :", r["raw"][:200])
            continue
        for champ in CHAMPS:
            if r["output"].get(champ) != r["expected"][champ]:
                print(f"  {champ} : attendu {r['expected'][champ]!r}, obtenu {r['output'].get(champ)!r}")


show_failures(EVALS[0])

## Comparaison avec un second modèle (facultatif)

Si `MODELE_2` est défini dans `.env`, exécutez le même test avec ce modèle et comparez la justesse, la latence et le coût. Sinon, passez cette étape.

In [ ]:
if os.getenv("MODELE_2"):
    EVALS.append(evaluate(os.environ["MODELE_2"]))
else:
    print("MODELE_2 non renseigné dans .env : étape ignorée.")
summarize(EVALS)

## <a id="formats"></a>Comparaison des formats

Le même prompt est exécuté sans `response_format` : le modèle reçoit seulement la consigne de produire du JSON. Le taux de rejet mesure les sorties qui ne peuvent pas être analysées ou qui ne respectent pas les types du schéma. Pydantic sert ici à valider les sorties après génération.

In [ ]:
def is_rejected(output):
    try:
        Reclamation.model_validate_json(json.dumps(output), strict=True)
        return False
    except ValidationError:
        return True


EVALS.append(evaluate(os.environ["MODELE"], label=f"{os.environ['MODELE']} (sans schéma)", structured=False))
table = summarize(EVALS)
table["Taux de rejet"] = [sum(is_rejected(r["output"]) for r in ev["results"]) / len(ev["results"]) for ev in EVALS]
table

## <a id="conclusion"></a>Conclusion

À partir des résultats, rédigez deux ou trois phrases. Indiquez si le modèle convient à l'assistant, le coût estimé pour 1 000 réclamations (en rappelant qu'il s'agit d'une extrapolation), le taux de rejet et une erreur qui pourrait être corrigée dans le prompt.

**TODO 4**

In [ ]:
CONCLUSION = ""

## Contrôle automatique

La dernière cellule affiche ÉCHEC tant que les `# TODO` ne sont pas faits.

In [ ]:
try:
    assert Reclamation.model_fields, "Reclamation : déclarez les cinq champs (TODO 1)"
    assert PROMPT.strip(), "PROMPT : écrivez le prompt (TODO 2)"
    res = EVALS[0]["results"]
    assert len(res) == 10 and all(isinstance(r["output"], dict) for r in res), "extract : chaque sortie doit être un dictionnaire (TODO 3)"
    total = sum(r["score"] for r in res)
    assert total >= 40, f"{total}/50 champs justes : améliorez le prompt (objectif : au moins 40)"
    assert len(CONCLUSION) >= 60 and any(c.isdigit() for c in CONCLUSION), "écrivez une conclusion avec au moins un chiffre du tableau (TODO 4)"
    print("OK")
except Exception as e:
    print(f"ÉCHEC : {e if isinstance(e, AssertionError) else 'terminez les # TODO et relancez les cellules.'}")